# ---------------------This file is for testing purpose----------------------------

In [67]:
## Importing all necessary libraries.

import os
import json
import gzip
import shutil
from datetime import datetime

import requests
from pandas import json_normalize
from dotenv import load_dotenv

In [68]:
## Calling load_dotenv function with override = True, because if in future API Key changes it will override the old values in this file automatically.
load_dotenv(override=True)

True

#### -------Defining all User Configuration Variables--------

In [70]:
# UN Comtrade subscription key
SUBSCRIPTION_KEY = os.getenv("UN_COMTRADE_PRIMARY_KEY")
#print(f"UN Comtrade Subscription Key: {SUBSCRIPTION_KEY}")

# Folder where raw .gz files and final combined file will be stored
OUTPUT_DIR = os.getenv("OUTPUT_DIRECTORY")
#print(f"Output Directory: {OUTPUT_DIR}")

# Year Range from which year to which year data is needed
START_YEAR = 2015
END_YEAR = 2024

# UN Comtrade FINAL dataset settings
TYPE_CODE = "C"      # Commodity trade / goods trade
FREQ_CODE = "A"      # Annual data
CL_CODE = "HS"       # HS classification

# Kept reporterCode = None to download ALL reporters i.e. to download world trade data
REPORTER_CODE = None

# If True, original downloaded .gz files will be deleted after combining
DELETE_RAW_GZ_AFTER_COMBINE = False

# Final combined file format:
# True  -> keep final combined file as .txt
# False -> compress final combined file into .gz after combining
KEEP_COMBINED_AS_TXT = True

# Request timeout settings (in seconds)
METADATA_TIMEOUT = 120
DOWNLOAD_TIMEOUT = 300

### Helper functions

In [71]:
## Helper Function to ensure output directory is available for storage
def ensure_directory(path: str):
    os.makedirs(path, exist_ok=True)

In [72]:
## Helper Function to create request session
def create_session():
    
    session = requests.Session()

    # Default headers for all requests made through this session
    session.headers.update({
        "User-Agent": "Comtrade-FINAL-Downloader/1.0"
    })

    return session

In [73]:
# ============================================================
# STEP 1: GET AVAILABLE FILE METADATA FROM COMTRADE BULK API
# ============================================================


def get_bulk_file_metadata(session,subscription_key, type_code, freq_code, cl_code, period, reporter_code=None):

    ## Final Bulk-dataset endpoint
    base_url = f"https://comtradeapi.un.org/bulk/v1/get/{type_code}/{freq_code}/{cl_code}"

    ## Query Params
    params = {
        "subscription-key": subscription_key,
        "period": period,
        "reportercode": reporter_code
    }

    params = {k: v for k, v in params.items() if v is not None}

    # Sending GET request to Comtrade API
    response = session.get(base_url, params=params, timeout=METADATA_TIMEOUT)

    try:
        response.raise_for_status()
    except requests.exceptions.HTTPError as e:
        raise Exception(
            f"Metadata request failed for year {period}. "
            f"HTTP Status: {response.status_code}. Response: {response.text}"
        ) from e
    
    result = response.json()

    if result.get("count") == 0:
        raise Exception(f"No metadata found for year {period}. Response: {result}")
        return []
    
    df = json_normalize(result["data"])

    # Convert DataFrame rows to list of dictionaries
    return df.to_dict(orient="records")

In [74]:
# ============================================================
# STEP 2: DOWNLOAD ONE FILE FROM ITS FILE URL
# ============================================================

def download_file(session, file_url, output_path, subscription_key):
    
    # Downloading one .gz file from Comtrade and saving it locally.

    params = {"subscription-key": subscription_key}

    with session.get(file_url, params=params, stream=True, timeout=DOWNLOAD_TIMEOUT) as response:
        try:
            response.raise_for_status()
        except requests.exceptions.HTTPError as e:
            raise Exception(
                f"File download failed. HTTP Status: {response.status_code} for URL: {file_url}"
            ) from e

        # Write downloaded file content chunk by chunk to disk
        with open(output_path, "wb") as out_file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):  # 1 MB chunks
                if chunk:
                    out_file.write(chunk)

In [75]:
# ============================================================
# STEP 3: BUILD A CLEAN FILE NAME FOR EACH DOWNLOADED FILE
# ============================================================

def build_file_name(record):

    ## Creates a clean local file name using Comtrade metadata fields.
    
    publication_date = record.get("publicationDate")
    publication_date = publication_date[:10] if publication_date else "1900-01-01"

    # Build a readable file name for the raw .gz file
    file_name = (
        f"COMTRADE-FINAL-"
        f"{record.get('typeCode', '')}"
        f"{record.get('freqCode', '')}"
        f"{str(record.get('reporterCode', '')).zfill(3)}"
        f"{record.get('period', '')}"
        f"{record.get('classificationCode', '')}"
        f"[{publication_date}].gz"
    )

    return file_name

In [76]:
# ============================================================
# STEP 4: DOWNLOAD ALL FINAL FILES FOR A LIST OF YEARS
# ============================================================

def download_comtrade_final_years(session, subscription_key, output_dir, start_year, end_year, type_code="C", freq_code="A", cl_code="HS", reporter_code=None):

    ensure_directory(output_dir)

    downloaded_files = []

    # Loop through each requested year
    for year in range(start_year, end_year + 1):
        print(f"\n========== Processing year {year} ==========")

        # Get metadata for all downloadable files for this year
        records = get_bulk_file_metadata(
            session=session,
            subscription_key=subscription_key,
            type_code=type_code,
            freq_code=freq_code,
            cl_code=cl_code,
            period=year,
            reporter_code=reporter_code
        )

        # If no files are available for that year, move to next year
        if not records:
            print(f"No files available for year {year}")
            continue

        print(f"Found {len(records)} file(s) for year {year}")

        # Download each file listed in metadata
        for record in records:
            file_url = record.get("fileUrl")
            if not file_url:
                continue

            file_name = build_file_name(record)
            output_path = os.path.join(output_dir, file_name)

            print(f"Downloading: {file_name}")
            download_file(session, file_url, output_path, subscription_key)

            downloaded_files.append(output_path)

        print(f"Completed downloads for year {year}")

    return downloaded_files

In [77]:
# ============================================================
# STEP 5: COMBINE ALL DOWNLOADED .GZ FILES INTO ONE MASTER FILE
# ============================================================

def combine_gz_files(gz_files, output_dir, start_year, end_year, type_code="C", freq_code="A", cl_code="HS", keep_txt=True, delete_raw_gz=False):

    if not gz_files:
        print("No downloaded .gz files found to combine.")
        return None

    # Create a timestamp so output file names are unique
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Build final combined file base name
    combined_base_name = (
        f"COMBINED-COMTRADE-FINAL-{type_code}{freq_code}{cl_code}"
        f"-WORLD-{start_year}_{end_year}-[{timestamp}]"
    )

    txt_output_path = os.path.join(output_dir, combined_base_name + ".txt")
    gz_output_path = os.path.join(output_dir, combined_base_name + ".gz")

    print("\n========== Combining downloaded files ==========")

    # Write combined output as a text file first
    with open(txt_output_path, "w", encoding="utf-8") as txt_out:
        header_written = False

        for gz_file in gz_files:
            print(f"Reading: {os.path.basename(gz_file)}")

            # Open each gz file in text mode
            with gzip.open(gz_file, "rt", encoding="utf-8") as in_file:
                for i, line in enumerate(in_file):
                    # First line of each file is assumed to be header
                    if i == 0:
                        # Write header only once from the first file
                        if not header_written:
                            txt_out.write(line)
                            header_written = True
                    else:
                        # Write all data rows
                        txt_out.write(line)

    # If user wants final output as compressed .gz instead of .txt
    if not keep_txt:
        with open(txt_output_path, "rb") as f_in:
            with gzip.open(gz_output_path, "wb") as f_out:
                shutil.copyfileobj(f_in, f_out)

        # Remove temporary .txt after creating final .gz
        os.remove(txt_output_path)

    # Optionally delete raw downloaded .gz files after combine
    if delete_raw_gz:
        for gz_file in gz_files:
            if os.path.exists(gz_file):
                os.remove(gz_file)


    # Return final output path
    final_output = txt_output_path if keep_txt else gz_output_path
    print(f"\nCombined master file created successfully:\n{final_output}")

    return final_output

In [ ]:
# ============================================================
# MAIN DRIVER FUNCTION
# ============================================================

def main():
    
    # Main pipeline:
    # 1. Download FINAL world trade data for selected years
    # 2. Combine all downloaded files into one master dataset
    
    print(SUBSCRIPTION_KEY)
    # Basic validation so you don't accidentally run without key
    if SUBSCRIPTION_KEY == "YOUR_COMTRADE_SUBSCRIPTION_KEY":
        raise ValueError("Please replace SUBSCRIPTION_KEY with your actual UN Comtrade subscription key.")

    ensure_directory(OUTPUT_DIR)

    print("Starting UN Comtrade FINAL bulk download pipeline...")
    print(f"Output directory: {OUTPUT_DIR}")
    print(f"Year range: {START_YEAR} to {END_YEAR}")
    print("Reporter scope: ALL reporters / world trade")
    print(f"Dataset: FINAL | Type={TYPE_CODE} | Frequency={FREQ_CODE} | Classification={CL_CODE}")

    # Create one reusable HTTP session for the full pipeline
    session = create_session()
    
    try:
        # Step 1: Download raw .gz files for all requested years
        downloaded_files = download_comtrade_final_years(
            session=session,
            subscription_key=SUBSCRIPTION_KEY,
            output_dir=OUTPUT_DIR,
            start_year=START_YEAR,
            end_year=END_YEAR,
            type_code=TYPE_CODE,
            freq_code=FREQ_CODE,
            cl_code=CL_CODE,
            reporter_code=REPORTER_CODE
        )

        print(f"\nTotal raw files downloaded: {len(downloaded_files)}")

        # Step 2: Combine downloaded files into one master file
        final_file = combine_gz_files(
            gz_files=downloaded_files,
            output_dir=OUTPUT_DIR,
            start_year=START_YEAR,
            end_year=END_YEAR,
            type_code=TYPE_CODE,
            freq_code=FREQ_CODE,
            cl_code=CL_CODE,
            keep_txt=KEEP_COMBINED_AS_TXT,
            delete_raw_gz=DELETE_RAW_GZ_AFTER_COMBINE
        )
        print("\nPipeline completed successfully.")
        print(f"Final combined dataset file: {final_file}")

    finally:
        # Close the session to release resources
        session.close()

# Run the pipeline only when this file is executed directly
if __name__ == "__main__":
    main()